# Tiebreak-Level Panel

Reshapes the team-level Grand Slam panel into a **tiebreak-level** panel where each
observation is one team in one specific tiebreak.

| Match type | Team obs | Tiebreak obs |
|---|---|---|
| 1 regular TB (set 1 or 2) | 2 | 2 |
| 2 regular TBs (both sets) | 2 | 4 |
| Match TB (10-pt set 3) | 2 | 2 |
| Set-3 regular TB (7-pt) | 2 | 2 |

Outcome (`won_tb`): did this team win this specific tiebreak?

Output: `data/atp/tiebreak_panel.csv`

In [1]:
import os
import pandas as pd
import numpy as np

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PANEL_PATH = os.path.join(ROOT, 'data', 'atp', 'team_gs_panel.csv')
OUT_PATH   = os.path.join(ROOT, 'data', 'atp', 'tiebreak_panel.csv')
print('Input: ', PANEL_PATH)
print('Output:', OUT_PATH)

Input:  C:\Users\ALESSANDRO\Documents\GitHub\tennis-homophily\data\atp\team_gs_panel.csv
Output: C:\Users\ALESSANDRO\Documents\GitHub\tennis-homophily\data\atp\tiebreak_panel.csv


## 1. Load team panel — same filters as main analysis

In [2]:
df = pd.read_csv(PANEL_PATH)
print(f'Loaded: {len(df)} team-obs ({len(df)//2} matches)')

# Exclude Olympics (same filter as Tables 3-5)
df = df[df['olympics_tourn'] == 0].copy()
print(f'After excl. Olympics: {len(df)} team-obs ({len(df)//2} matches)')

# Drop rows missing GS experience (same filter as Tables 3-5)
df = df[df['exp_mean'].notna()].copy()
print(f'After exp_mean filter: {len(df)} team-obs')

# Keep only complete match pairs - both team rows must survive all filters.
# Orphaned rows arise when only one team's exp_mean is available; they are fine
# in the match-win regression but break the tiebreak panel's 2-rows-per-TB structure.
pair_size = df.groupby('match_id')['match_id'].transform('count')
df = df[pair_size == 2].copy()
print(f'After requiring complete pairs: {len(df)} team-obs ({len(df)//2} matches)')

# Generate exp_mean_sq if not already present
if 'exp_mean_sq' not in df.columns:
    df['exp_mean_sq'] = df['exp_mean'] ** 2

Loaded: 3744 team-obs (1872 matches)
After excl. Olympics: 3744 team-obs (1872 matches)
After exp_mean filter: 3744 team-obs
After requiring complete pairs: 3744 team-obs (1872 matches)


## 2. Derive per-set tiebreak flags

The team panel has `won_tb_s1` and `won_tb_s2` per team-row. For each match, exactly one
team has `won_tb_s1 = 1` if there was a set-1 tiebreak (the other has 0); if there was no
set-1 tiebreak both teams have 0.  We recover the per-match flag via the max per match.

In [3]:
# Per-match flags: was there a tiebreak in each specific set?
df['has_tb_s1'] = df.groupby('match_id')['won_tb_s1'].transform('max').gt(0).astype(int)
df['has_tb_s2'] = df.groupby('match_id')['won_tb_s2'].transform('max').gt(0).astype(int)
# match_tb and tb_s3_regular are already match-level flags in the panel

# For set-3 TBs the match winner (win=1) is always the tiebreak winner
# because the last set determines who advances.
df['won_match_tb']    = ((df['match_tb']      == 1) & (df['win'] == 1)).astype(int)
df['won_tb_s3_reg']   = ((df['tb_s3_regular'] == 1) & (df['win'] == 1)).astype(int)

print('Matches with set-1 TB:          ', df.groupby('match_id')['has_tb_s1'].first().sum())
print('Matches with set-2 TB:          ', df.groupby('match_id')['has_tb_s2'].first().sum())
print('Matches with 10-pt match TB:    ', df.groupby('match_id')['match_tb'].first().sum())
print('Matches with set-3 regular TB:  ', df.groupby('match_id')['tb_s3_regular'].first().sum())

Matches with set-1 TB:           463
Matches with set-2 TB:           470
Matches with 10-pt match TB:     4
Matches with set-3 regular TB:   211


## 3. Reshape to tiebreak-level

For each team-row, we create one tiebreak observation per tiebreak type present in that
match. The result is a long dataframe with columns `tb_set` (1/2/3) and `tb_type`
(7pt / 10pt) alongside all original team-level variables.

In [4]:
base_cols = [
    'match_id', 'tournament', 'year', 'surface', 'stage_code',
    'same_country', 'same_language', 'ling_prox',
    'rank_mean', 'opp_rank_mean', 'rank_gap', 'single_top100',
    'exp_mean', 'exp_mean_sq', 'win',
    'pre_olympic', 'olympic_period',
]

# Each entry: (filter_col, won_col, tb_set, tb_type)
TB_DEFS = [
    ('has_tb_s1',      'won_tb_s1',      1, '7pt'),
    ('has_tb_s2',      'won_tb_s2',      2, '7pt'),
    ('tb_s3_regular',  'won_tb_s3_reg',  3, '7pt'),
    ('match_tb',       'won_match_tb',   3, '10pt'),
]

chunks = []
for flag_col, won_col, tb_set, tb_type in TB_DEFS:
    sub = df[df[flag_col] == 1][base_cols + [won_col]].copy()
    sub = sub.rename(columns={won_col: 'won_tb'})
    sub['tb_set']  = tb_set
    sub['tb_type'] = tb_type
    chunks.append(sub)
    print(f'  Set-{tb_set} {tb_type} TB: {len(sub)} team-obs ({len(sub)//2} tiebreaks)')

tb_panel = pd.concat(chunks, ignore_index=True)
tb_panel = tb_panel.sort_values(['match_id', 'tb_set', 'tb_type', 'win']).reset_index(drop=True)

print(f'\nTotal tiebreak team-obs: {len(tb_panel)}')
print(f'Unique tiebreaks (match×set×type): {len(tb_panel)//2}')

  Set-1 7pt TB: 926 team-obs (463 tiebreaks)
  Set-2 7pt TB: 940 team-obs (470 tiebreaks)
  Set-3 7pt TB: 422 team-obs (211 tiebreaks)
  Set-3 10pt TB: 8 team-obs (4 tiebreaks)

Total tiebreak team-obs: 2296
Unique tiebreaks (match×set×type): 1148


## 4. Sanity checks

In [5]:
# Each (match_id, tb_set, tb_type) should have exactly 2 rows (one per team)
counts = tb_panel.groupby(["match_id", "tb_set", "tb_type"]).size()
incomplete = counts[counts != 2]
if len(incomplete) == 0:
    print("OK: every tiebreak has exactly 2 team rows")
else:
    print(f"WARNING: {len(incomplete)} tiebreak groups with != 2 rows")
    print(incomplete)

# won_tb should sum to 1 per tiebreak for complete pairs
won_sums = tb_panel.groupby(["match_id", "tb_set", "tb_type"])["won_tb"].sum()
assert (won_sums == 1).all(), f"won_tb sum != 1 for some tiebreaks: {won_sums[won_sums != 1]}"
print("OK: exactly one winner per tiebreak")

# Missing values in key regression variables
key_vars = ["won_tb", "same_country", "same_language", "ling_prox",
            "rank_mean", "opp_rank_mean", "exp_mean"]
missing_counts = tb_panel[key_vars].isna().sum()
print("")
print("Missing values in key variables:")
print(missing_counts[missing_counts > 0] if missing_counts.any() else "  None")

print("")
print("Tiebreak obs by type:")
print(tb_panel.groupby(["tb_set", "tb_type"]).size().rename("n_team_obs"))

OK: every tiebreak has exactly 2 team rows
OK: exactly one winner per tiebreak

Missing values in key variables:
  None

Tiebreak obs by type:
tb_set  tb_type
1       7pt        926
2       7pt        940
3       10pt         8
        7pt        422
Name: n_team_obs, dtype: int64


## 5. Save

In [6]:
tb_panel.to_csv(OUT_PATH, index=False)
print(f'Saved {len(tb_panel)} rows → {OUT_PATH}')
print()
print('Column list:')
print(tb_panel.columns.tolist())
print()
print('Sample rows:')
tb_panel.head(6)

Saved 2296 rows → C:\Users\ALESSANDRO\Documents\GitHub\tennis-homophily\data\atp\tiebreak_panel.csv

Column list:
['match_id', 'tournament', 'year', 'surface', 'stage_code', 'same_country', 'same_language', 'ling_prox', 'rank_mean', 'opp_rank_mean', 'rank_gap', 'single_top100', 'exp_mean', 'exp_mean_sq', 'win', 'pre_olympic', 'olympic_period', 'won_tb', 'tb_set', 'tb_type']

Sample rows:


,match_id,tournament,year,surface,stage_code,same_country,same_language,ling_prox,rank_mean,opp_rank_mean,rank_gap,single_top100,exp_mean,exp_mean_sq,win,pre_olympic,olympic_period,won_tb,tb_set,tb_type
0,2,Australian Open,2018,Hard,2,1.0,1.0,1.0,87.5,55.5,13.0,0,5.0,25.00,0,0,0,1,1,7pt
1,2,Australian Open,2018,Hard,2,0.0,0.0,0.0,55.5,87.5,83.0,0,14.5,210.25,1,0,0,0,1,7pt
2,3,Australian Open,2018,Hard,2,0.0,0.0,0.0,472.5,35.0,749.0,0,11.0,121.00,0,0,0,0,1,7pt
3,3,Australian Open,2018,Hard,2,0.0,1.0,1.0,35.0,472.5,26.0,0,14.0,196.00,1,0,0,1,1,7pt
4,7,Australian Open,2018,Hard,2,0.0,1.0,1.0,20.0,79.5,10.0,0,12.5,156.25,0,0,0,0,1,7pt
5,7,Australian Open,2018,Hard,2,0.0,0.0,1.0,79.5,20.0,5.0,0,16.0,256.00,1,0,0,1,1,7pt
